# Phase 12 — Jacobi-Linse v2

v1 hat die Gleichwertigkeit des Transports bewiesen (4e-7 relativ) und dass das
Modell differenzierbar ist — und ist dann daran gestorben, dass der
Flash-Attention-Kernel **keine zweite Ableitung** kann, und an OOM bei Stapel 5.

v2 stellt auf eager um und prüft die zweite Ableitung mit einem billigen
Mini-Aufruf; bei Speichermangel wird der Stapel global halbiert und der Prompt
wiederholt, statt still Zeilen auf null zu lassen. Als Ausweichweg dieselbe
Größe als zentrale Differenz — zwei Vorwärtspässe, kein Autograd, läuft durch
jeden Kernel, mit gemessener Linearitätsprobe.

Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~12 min.

In [ ]:
# === JACOBI-LINSE v2: zwei Abbrueche aus v1 behoben =======================
# v1 hat zwei Dinge bewiesen und ist dann an zwei anderen gestorben.
# Bewiesen:
#   - Gleichwertigkeit unseres Transports zum Upstream-Schaetzer:
#     max|JVP - J@v| = 9.5e-07 bei Skala 2.45, also 4e-7 relativ.
#   - Das Modell ist differenzierbar (|grad|=4.34). FP8 spielt keine Rolle,
#     transformers dequantisiert auf bf16, weil die Karte Rechenfaehigkeit 8.0
#     hat. Der erste Rueckwaertspass laeuft also durch 37 Schichten.
# Gestorben an:
#   (1) RuntimeError: derivative for aten::_scaled_dot_product_flash_attention_
#       backward is not implemented. Der Flash-Kernel kann EINE Ableitung, aber
#       keine ZWEITE - und der Doppel-Rueckwaerts-Trick braucht genau die.
#       -> v2 stellt vorher auf eager um (reine Torch-Operationen, zweifach
#          differenzierbar) und PRUEFT das mit einem billigen Mini-Aufruf,
#          statt es zu hoffen.
#   (2) OOM schon bei Stapel 5. Der zurueckbehaltene Graph mit create_graph
#          kostet doppelt.
#       -> v2 startet bei Stapel 2 und 64 Tokens, halbiert bei Speichermangel
#          GLOBAL und wiederholt den Prompt. Teilergebnisse werden erst
#          uebernommen, wenn alle Positionen durch sind - sonst blieben
#          einzelne Zeilen stumm auf null stehen, ohne dass es auffaellt.
#
# Und ein Ausweichweg, falls auch eager die zweite Ableitung nicht hergibt
# (die gated-DeltaNet-Schichten laufen im Torch-Fallback): dieselbe Groesse
# als ZENTRALE DIFFERENZ - Residuum an allen gueltigen Quellpositionen um
# +-eps*v stoeren, zwei Vorwaertspaesse, Differenz. Kein Gradient, kein Graph,
# laeuft durch jeden Kernel. Der Preis ist Linearitaet plus bf16-Rundung, und
# genau das wird gemessen: eps gegen 2*eps, relativer Abstand wird gedruckt.
# Der Gleichwertigkeitsbeweis am Anfang prueft BEIDE Wege gegen die wirklich
# aufgestellte Matrix aus jacobian_for_prompt.
#
# Vorregistrierung Nr. 28 bleibt: KEINE ~40%, RICHTUNG ~35%, BEIDE ~25%.
# Selbstversorgend, FRISCHE Runtime, einzige Zelle. ~12 min.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","lauf")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

import matplotlib.pyplot as plt
import subprocess, sys
N_FIT=24; MAXTOK=64; BATCH=2; TOPK=6; SEED=0; EPS=0.1
LAYERS=[3,7,11,15,19,23,27,31,35]     # Voll-Attention-Layer als Quellschichten
N_CTRL=15
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- reine Hilfslogik (offline testbar) ------------------------
def sample_positions(lo,hi,n,must,seed=0):
    if hi<=lo: return sorted(set(must))
    step=max(1,(hi-lo)//max(1,n))
    return sorted(set(list(range(lo,hi+1,step))[:n]+[t for t in must if lo<=t<=hi]))
def chunks(xs,k): return [xs[i:i+k] for i in range(0,len(xs),k)]
def rank_of(vals,idx):
    v=list(vals); t=v[idx]; return 1+sum(1 for x in v if x>t)
def linearity_error(a,b):
    """a bei eps, b bei 2*eps - beide schaetzen dieselbe Groesse. Relativer
       Abstand ist das Mass fuer Nichtlinearitaet plus Rundungsrauschen."""
    d=float(np.abs(np.asarray(a)-np.asarray(b)).max())
    s=float(np.abs(np.asarray(a)).max())
    return d/max(s,1e-30)
def verdict_jlens(rank_q,n_pos,logit_rank_q):
    top=rank_q<=max(2,n_pos//10)
    if not top: return "KEINE"
    if logit_rank_q<=max(2,n_pos//10): return "BEIDE"
    return "RICHTUNG"
# ---------------- jlens beschaffen -----------------------------------------
JL="/content/jacobian_lens"
if not os.path.isdir(os.path.join(JL,"jlens")):
    r=subprocess.run(["git","clone","--depth","1",
                      "https://github.com/Erikiss/jacobian-lens",JL],
                     capture_output=True,text=True,timeout=600)
    assert r.returncode==0, "Klon fehlgeschlagen: %s"%r.stderr.strip()[-200:]
subprocess.run([sys.executable,"-m","pip","install","-q","--no-deps","-e",JL],
               capture_output=True,text=True)
if JL not in sys.path: sys.path.insert(0,JL)
from jlens.hooks import ActivationRecorder
from jlens.fitting import jacobian_for_prompt, valid_position_mask
print("jlens geladen aus", JL)
# ---------------- Weg A: Doppel-Rueckwaerts (exakt, braucht 2. Ableitung) ---
def jvp_transport(fwd,blocks,input_ids,source_layers,target_layer,V,skip_first,B):
    """J_l @ v ohne J_l aufzustellen:
           g(s) = d(y.s)/du = J^T s   (Rueckwaerts mit create_graph)
           d(g.v)/ds        = J v     (zweiter Rueckwaerts)
       Braucht eine ZWEITE Ableitung durch die Attention - der Flash-Kernel
       hat die nicht (aten::_scaled_dot_product_flash_attention_backward),
       deshalb muss vorher auf eager umgestellt werden."""
    with ActivationRecorder(blocks,at=[*source_layers,target_layer],
                            start_graph_at=min(source_layers)) as rec, torch.enable_grad():
        ids=input_ids.expand(B,-1)
        fwd(ids)
        y=rec.activations[target_layer]
        us=[rec.activations[l] for l in source_layers]
        pm=valid_position_mask(ids.shape[1],skip_first=skip_first)
        pos=pm.nonzero(as_tuple=True)[0].to(y.device); npos=int(pm.sum())
        s=torch.zeros_like(y,requires_grad=True)
        gs=torch.autograd.grad(y,us,grad_outputs=s,create_graph=True)
        out={}
        for i,l in enumerate(source_layers):
            t=torch.zeros_like(us[i]); p2=pos.to(t.device)
            t[:,p2,:]=V[l].to(t.dtype).to(t.device)[:,None,:]
            jv,=torch.autograd.grad(gs[i],s,grad_outputs=t,
                                    retain_graph=(i<len(source_layers)-1))
            out[l]=jv[:,pos,:].float().sum(1)/npos
        return out
# ---------------- Weg B: zentrale Differenz (kein Autograd noetig) ---------
@torch.no_grad()
def fd_transport(fwd,blocks,input_ids,source_layers,target_layer,V,skip_first,B,eps=0.1):
    """Dieselbe Groesse als zentrale Differenz: das Residuum an allen gueltigen
       Quellpositionen um +-eps*v stoeren, zwei Vorwaertspaesse, Differenz.
       Kein Gradient, kein Graph - laeuft durch jeden Kernel. Preis: der
       Schaetzer ist nur bis zur Linearitaet exakt, und bf16 rundet mit. Beides
       wird unten gemessen (Linearitaetsprobe eps gegen 2*eps)."""
    ids=input_ids.expand(B,-1)
    pm=valid_position_mask(ids.shape[1],skip_first=skip_first)
    npos=int(pm.sum())
    st={"l":None,"sign":0,"tan":None,"pos":None}
    def mk(idx):
        def hook(mod,inp,out):
            if st["l"]!=idx or st["sign"]==0: return None
            t=out if torch.is_tensor(out) else out[0]
            t2=t.clone()
            p=st["pos"].to(t2.device)
            t2[:,p,:]=t2[:,p,:]+ (st["sign"]*eps)*st["tan"].to(t2.dtype).to(t2.device)[:,None,:]
            return t2 if torch.is_tensor(out) else (t2,)+tuple(out[1:])
        return hook
    handles=[blocks[l].register_forward_hook(mk(l)) for l in source_layers]
    out={}
    try:
        with ActivationRecorder(blocks,at=[target_layer]) as rec:
            def run():
                fwd(ids)
                a=rec.activations[target_layer]
                return a[:,st["pos"].to(a.device),:].float().sum(1)
            st["pos"]=pm.nonzero(as_tuple=True)[0]
            for l in source_layers:
                st["l"]=l; st["tan"]=V[l]
                st["sign"]=1;  yp=run()
                st["sign"]=-1; ym=run()
                st["l"]=None; st["sign"]=0
                out[l]=(yp-ym)/(2.0*eps*npos)
    finally:
        for h in handles: h.remove()
    return out
# ---------------- Gleichwertigkeitsbeweis gegen den Upstream ---------------
from tests.tiny import TinyDecoder
_t=TinyDecoder(n_layers=5,d_model=8,seed=0).eval()
for _p in _t.parameters(): _p.requires_grad_(False)
_TXT="the quick brown fox jumps over the lazy dog again and again"
_SL=[0,1,2]; _SK=2
_J,_sq,_nv=jacobian_for_prompt(_t,_TXT,_SL,dim_batch=4,max_seq_len=64,skip_first=_SK)
_ids=_t.encode(_TXT,max_length=64)
torch.manual_seed(0); _V={l:torch.randn(3,8) for l in _SL}
_REF={l:_V[l]@_J[l].T for l in _SL}
_sc=max(float(_REF[l].abs().max()) for l in _SL)
_A=jvp_transport(lambda x:_t(x),_t.layers,_ids,_SL,4,_V,_SK,3)
_B=fd_transport(lambda x:_t(x),_t.layers,_ids,_SL,4,_V,_SK,3,eps=1e-3)
_dA=max(float((_A[l]-_REF[l]).abs().max()) for l in _SL)
_dB=max(float((_B[l]-_REF[l]).abs().max()) for l in _SL)
print("Gleichwertigkeit zum Upstream-Schaetzer (Skala %.3e):"%_sc)
print("  Doppel-Rueckwaerts  max|JVP - J@v| = %.3e"%_dA)
print("  zentrale Differenz  max|FD  - J@v| = %.3e"%_dB)
assert _dA<1e-4*max(_sc,1.0) and _dB<1e-2*max(_sc,1.0), "Transport stimmt nicht - Abbruch"
print("  -> beide Wege liefern denselben Transport wie die aufgestellte Matrix.")
del _t,_J,_A,_B; gc.collect()
# ---------------- eager erzwingen, dann Weg waehlen ------------------------
BLOCKS=model.model.layers
for _p in model.parameters(): _p.requires_grad_(False)
def fwd(ids): return model.model(input_ids=ids)
_UD=next(model.model.norm.parameters()).dtype
def unembed(r): return model.lm_head(model.model.norm(r.to(_UD)))
ATTN_OK=False
try:
    model.set_attn_implementation("eager"); ATTN_OK=True
except Exception:
    try: model.config._attn_implementation="eager"; ATTN_OK=True
    except Exception: pass
print("\nAttention-Implementierung auf eager gestellt: %s"%("ja" if ATTN_OK else "nein"))
def probe_second_derivative():
    """genau der Aufruf, der im letzten Lauf gestorben ist - klein und billig"""
    ii=tokenizer("a short probe sentence for the second derivative check",
                 return_tensors="pt").input_ids.to(model.device)
    V={3:torch.randn(1,model.config.hidden_size)*0.0}
    jvp_transport(fwd,BLOCKS,ii,[3],model.config.num_hidden_layers-1,V,4,1)
MODE="fd"
try:
    probe_second_derivative(); MODE="jvp"
    print("  zweite Ableitung laeuft durch -> exakter Weg (Doppel-Rueckwaerts)")
except Exception as e:
    print("  zweite Ableitung nicht verfuegbar (%s: %s)"%(type(e).__name__,str(e)[:90]))
    print("  -> Ausweichweg: zentrale Differenz, zwei Vorwaertspaesse je Layer")
gc.collect(); torch.cuda.empty_cache()
# ---------------- Zielprompt und Positionen --------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
pre=think_prefix(TAB)
enc=tokenizer(pre,return_offsets_mapping=True); IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=[i for i,(a,b) in enumerate(enc["offset_mapping"]) if b>c0 and a<c1 and b>a]
K,Q=DEC[0],DEC[-1]
UT=[i for i,(a,b) in enumerate(enc["offset_mapping"])
    if b>len(SCAFF) and a<len(SCAFF)+len(TAB) and b>a]
POS=sample_positions(UT[0],UT[-1],N_CTRL,[K-1,K,Q,Q+1],seed=SEED)
jQ=POS.index(Q)
print("\nZielprompt %d Tokens | Koeder K=%d %r  Q=%d %r | %d Lesepositionen"
      %(L,K,tokenizer.decode([IDS[K]]),Q,tokenizer.decode([IDS[Q]]),len(POS)))
ids_t=torch.tensor([IDS],device=model.device)
with torch.no_grad():
    HS=model(input_ids=ids_t,output_hidden_states=True).hidden_states
H={l:torch.stack([HS[l+1][0,p] for p in POS]).float().cpu() for l in LAYERS}
del HS; gc.collect(); torch.cuda.empty_cache()
# ---------------- Korpus -----------------------------------------------------
rng=np.random.default_rng(SEED)
corp=[p for p in PROMPT_IDS if 200<len(PROMPTS[p])<=1200]
corp=[PROMPTS[corp[i]] for i in rng.permutation(len(corp))[:N_FIT]]
print("Korpus fuer die Mittelung: %d Prompts, je bis %d Tokens"%(len(corp),MAXTOK))
# ---------------- Transport mit Speicher-Ruecknahme ------------------------
TGT=model.config.num_hidden_layers-1
DIM=model.config.hidden_size
ACC={l:torch.zeros(len(POS),DIM) for l in LAYERS}
def transport(cid,idxs,eps=EPS):
    V={l:H[l][idxs].to(model.device) for l in LAYERS}
    f=jvp_transport if MODE=="jvp" else fd_transport
    kw={} if MODE=="jvp" else {"eps":eps}
    return f(fwd,BLOCKS,cid,LAYERS,TGT,V,16,len(idxs),**kw)
# Bei Speichermangel wird der Stapel global halbiert und der Prompt WIEDERHOLT.
# Teilergebnisse werden erst uebernommen, wenn alle Positionen durch sind -
# sonst blieben einzelne Zeilen stumm auf null stehen, ohne dass es auffaellt.
NOK=0; BS=BATCH; LIN=[]; ci=0
while ci<len(corp):
    cid=tokenizer(corp[ci],return_tensors="pt",truncation=True,
                  max_length=MAXTOK).input_ids.to(model.device)
    if cid.shape[1]<24: ci+=1; continue
    try:
        tmp={l:torch.zeros(len(POS),DIM) for l in LAYERS}
        for grp in chunks(list(range(len(POS))),BS):
            out=transport(cid,grp)
            for l in LAYERS: tmp[l][grp]=out[l].cpu()
            if MODE=="fd" and not LIN:
                o2=transport(cid,grp,eps=2*EPS)
                LIN.append(max(linearity_error(out[l],o2[l]) for l in LAYERS)); del o2
            del out
        for l in LAYERS: ACC[l]+=tmp[l]
        NOK+=1; ci+=1
    except torch.cuda.OutOfMemoryError:
        del tmp
        gc.collect(); torch.cuda.empty_cache()
        if BS>1:
            BS=max(1,BS//2)
            print("  Speicher knapp bei Prompt %d - Stapel global auf %d, Prompt wiederholt"%(ci,BS))
            continue
        print("  Speicher reicht auch bei Stapel 1 nicht - Prompt %d uebersprungen"%ci)
        ci+=1
    gc.collect(); torch.cuda.empty_cache()
    if ci%6==0: print("  %2d/%d Korpus-Prompts (%d brauchbar, Stapel %d)"%(ci,len(corp),NOK,BS))
assert NOK>0, ("kein einziger Korpus-Prompt ging durch - Speicher zu knapp. "
               "MAXTOK oder BATCH kleiner setzen und erneut laufen lassen.")
for l in LAYERS: ACC[l]/=NOK
print("  gemittelt ueber %d Prompts (Weg: %s)"%(NOK,MODE))
if LIN:
    print("  Linearitaetsprobe (eps gegen 2*eps): %.3f relativer Abstand %s"
          %(max(LIN),"- unbedenklich" if max(LIN)<0.15 else "- !! eps zu gross oder bf16-Rauschen"))
# ---------------- Auslesen ---------------------------------------------------
assert MASK_NPZ and os.path.exists(MASK_NPZ), "vocab_foreign_masks.npz nicht gefunden"
M_script=torch.tensor(np.load(MASK_NPZ)["script"])
def fmass(lg):
    p=torch.softmax(lg.float(),-1); V=p.shape[-1]
    m=M_script.to(p.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    return p[...,m[:V]].sum(-1)
JAC=np.zeros((len(LAYERS),len(POS))); LOG=np.zeros_like(JAC); TOPS={}
with torch.no_grad():
    for i,l in enumerate(LAYERS):
        lj=unembed(ACC[l].to(model.device)); ll=unembed(H[l].to(model.device))
        JAC[i]=fmass(lj).cpu().numpy(); LOG[i]=fmass(ll).cpu().numpy()
        TOPS[("jac",l)]=[tokenizer.decode([t]) for t in lj[jQ].topk(TOPK).indices.tolist()]
        TOPS[("log",l)]=[tokenizer.decode([t]) for t in ll[jQ].topk(TOPK).indices.tolist()]
        del lj,ll
print("\nWas die Jacobi-Linse am Koeder-Token Q=%d liest (Top-%d):"%(Q,TOPK))
for l in LAYERS: print("  L%-2d  %s"%(l," | ".join(repr(t) for t in TOPS[("jac",l)])))
print("\nLogit-Linse an derselben Stelle:")
for l in LAYERS: print("  L%-2d  %s"%(l," | ".join(repr(t) for t in TOPS[("log",l)])))
BEST=int(np.argmax(JAC[:,jQ])); rq=rank_of(JAC[BEST],jQ); rl=rank_of(LOG[BEST],jQ)
print("\nFremdschrift-Masse (staerkste Jacobi-Schicht L%d):"%LAYERS[BEST])
print("  Jacobi: Q=%.4f | Median ueber %d Positionen %.4f | Rang von Q: %d"
      %(JAC[BEST,jQ],len(POS),float(np.median(JAC[BEST])),rq))
print("  Logit : Q=%.4f | Median %.4f | Rang von Q: %d"
      %(LOG[BEST,jQ],float(np.median(LOG[BEST])),rl))
# ---------------- Karten ------------------------------------------------------
fig,axs=plt.subplots(1,2,figsize=(15,4.4))
for ax,(M,name) in zip(axs,[(JAC,"Jacobi-Linse"),(LOG,"Logit-Linse")]):
    im=ax.imshow(np.log10(np.maximum(M,1e-12)),aspect="auto",cmap="magma",origin="lower")
    ax.set_yticks(range(len(LAYERS))); ax.set_yticklabels(["L%d"%l for l in LAYERS],fontsize=8)
    ax.set_xticks(range(len(POS)))
    ax.set_xticklabels(["%d %s"%(p,tokenizer.decode([IDS[p]]).strip()[:7]) for p in POS],
                       rotation=90,fontsize=7)
    ax.axvline(jQ,color="#22D3EE",lw=1.6)
    ax.set_title("%s: log10 Fremdschrift-Masse\n(tuerkis = Koeder-Token Q=%d)"%(name,Q),fontsize=10)
    plt.colorbar(im,ax=ax,fraction=.03,pad=.02)
plt.tight_layout(); plt.show()
# ---------------- Verdikt -----------------------------------------------------
code=verdict_jlens(rq,len(POS),rl)
print("\nVERDIKT:",end=" ")
if code=="RICHTUNG":
    print("VERBALISIERBAR: die Jacobi-Linse liest am Koeder-Token eine Fremdschrift-")
    print("  Disposition (Rang %d von %d bei L%d), die Logit-Linse dort nicht"%(rq,len(POS),LAYERS[BEST]))
    print("  (Rang %d). Der Zustand ist darauf gerichtet, etwas zu SAGEN, was im"%rl)
    print("  Residuum selbst noch nicht steht - ein Zeuge, den das Modell nicht")
    print("  schreibt und darum nicht abkoppeln kann.")
elif code=="BEIDE":
    print("BEIDE LINSEN: die Disposition steht schon im Residuum (Logit-Rang %d),"%rl)
    print("  der Transport bestaetigt sie (Rang %d), fuegt aber nichts hinzu."%rq)
else:
    print("KEIN AUSSCHLAG AM KOEDER: Q ragt in keiner Linse heraus (Jacobi-Rang %d,"%rq)
    print("  Logit-Rang %d von %d). An dieser Position ist die Disposition nicht"%(rl,len(POS)))
    print("  als Vokabular lesbar - passend zu Cell 19 und zum RDX-Befund.")
print("(Deskriptiv, ein Zielprompt: die Raenge vergleichen Positionen INNERHALB")
print(" desselben Prompts, es ist kein Test ueber Prompts hinweg.)")
JLENS_RESULTS=dict(verdict=code,mode=MODE,eager=ATTN_OK,layers=LAYERS,positions=POS,
                   Q=Q,K=K,jac=JAC.tolist(),logit=LOG.tolist(),n_corpus=NOK,
                   rank_jac=rq,rank_logit=rl,best_layer=LAYERS[BEST],
                   linearity=max(LIN) if LIN else None,
                   tops={"%s_L%d"%(a,b):v for (a,b),v in TOPS.items()})
wc_save_all()
